# Task 1 — Inductive Biases and Feature Representations (STL-10)

Runs Steps 1–6 of the assignment end to end. Every number is written to `task1/results/` (JSON + CSV) and every figure to `report/figures/task1_*.{png,pdf}`.

**Design choices (fixed before looking at any result):**

| Choice | Setting | Metric used to interpret it |
|---|---|---|
| Dataset | STL-10 (10 classes, 96×96), resized directly to the common 224×224 RGB image (no centre crop, which would cut an already tight object crop) | — |
| Additional colour intervention | **Luma-preserving hue rotation by 180°** (rotate the chroma vector in YIQ space; Y is untouched). Changes hue identity, keeps luminance edges, shading, texture contrast and saturation. Grayscale removes chroma; hue rotation gives *misleading* chroma. | Accuracy change vs. clean, prediction consistency |
| Cue-conflict pairs | 6 unordered pairs, each crossing animal/vehicle: airplane–cat, bird–car, dog–ship, deer–truck, horse–airplane, monkey–car; both directions | Shape bias **with** coverage; per-pair breakdown |
| Style strength | AdaIN α = 1.0 (full style statistics; public pretrained AdaIN decoder); style image = central 60 % crop of a test image of the style class outside the evaluation subset | Shape bias / coverage; supplementary sweep α ∈ {0.5, 0.75, 1.0} |
| Rejection rule | Pixel-only, fixed before evaluation: reject if pixel std < 0.04 (flat), mean absolute difference to the content < 0.05 (style failed), or smoothed-edge-map correlation with the content < 0.45 (shape destroyed; threshold set from a pilot batch of stylizations inspected by eye, sorted by this metric, without any model prediction). All 50 content images per direction are candidates; the first 20 accepted per direction are kept → 240 images | Accepted / rejected counts by reason |
| Visualization | t-SNE, cosine metric, perplexity 30, PCA init, seed 6304; one projection per (backbone, intervention) fitted on clean + transformed features | Class separation, mixing of clean/transformed points |

Supplementary (cheap, clearly marked): hue-angle sweep, patch-grid sweep (2×2, 8×8), and a random-pair cosine floor so that representation stability is comparable across backbones.

Set `QUICK_RUN = True` for a fast dry run on a small subset (not for reporting).

In [1]:
import sys, json, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

from common.seed import set_seed
from common.paths import DATA_ROOT, results_dir, cache_dir, checkpoints_dir, figure_path
from common.logging import save_table, save_json

SEED = 6304
QUICK_RUN = False            # True = tiny dry run (NOT for the report)
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS, CACHE, CKPT = results_dir(1), cache_dir(1), checkpoints_dir(1)
set_seed(SEED)
pd.set_option("display.precision", 4)
pd.set_option("display.width", 200)
print("device:", DEVICE, "| torch", torch.__version__)

device: cuda | torch 2.11.0+cu128


## Step 0 — Data, splits and the fixed 500-image evaluation subset (seed 6304)

In [2]:
from task1.data.stl10 import load_stl10, stl10_pil, to_common_224, Common224Dataset, pil_list_to_tensor
from task1.data.make_subset import stratified_train_val_split, class_balanced_test_subset, save_selection

train_ds, test_ds, CLASSES = load_stl10(DATA_ROOT)
N_CLASSES = len(CLASSES)
train_labels, test_labels = np.asarray(train_ds.labels), np.asarray(test_ds.labels)

train_idx, val_idx = stratified_train_val_split(train_labels, val_fraction=0.2, seed=SEED)
subset_idx, balance_report = class_balanced_test_subset(test_labels, n_total=100 if QUICK_RUN else 500, seed=SEED)
if QUICK_RUN:
    train_idx, val_idx = train_idx[::10], val_idx[::10]

save_selection(RESULTS / "selected_test_subset.json", subset_idx, test_labels[subset_idx], balance_report)
save_json({"train": train_idx.tolist(), "val": val_idx.tolist(), "seed": SEED}, RESULTS / "train_val_split.json")

y = test_labels[subset_idx]
clean_pils = [to_common_224(stl10_pil(test_ds, int(i))) for i in subset_idx]
print("classes:", CLASSES)
print(f"train {len(train_idx)} | val {len(val_idx)} | eval subset {len(subset_idx)}")
print("per-class counts in subset:", np.bincount(y, minlength=N_CLASSES))

classes: ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck']
train 4000 | val 1000 | eval subset 500
per-class counts in subset: [50 50 50 50 50 50 50 50 50 50]


## Frozen backbones and linear heads

Each backbone is frozen and its representation for every training/validation image is extracted once and cached (no augmentation is used, so this is identical to running the frozen backbone every epoch). One linear head per backbone: AdamW, lr 1e-3, weight decay 1e-4, batch 64, ≤ 50 epochs, early stopping after 5 epochs without a validation-accuracy improvement, seed 6304.

Each head standardizes its input with the training split's per-dimension mean and standard deviation (added in run v2). This is an affine map, so the head is still a linear classifier on the unchanged representation; Step 6 uses the raw representations. Without it (run v1) the CLIP head, whose input is a unit-norm embedding, stopped with logits about 20× too small (98.4 % accuracy at 0.59 mean confidence, validation loss 0.35 vs. 0.05–0.06 for the other heads). The v1 histories are in `task1/results/diagnostics/`.

In [3]:
from torch.utils.data import DataLoader
from task1.models.backbones import BACKBONES, FrozenBackbone, extract_features_from_loader, train_linear_head, LinearHead

backbones = {name: FrozenBackbone(name).to(DEVICE) for name in BACKBONES}

def cached_features(name, split, indices):
    path = CACHE / f"feats_{name}_{split}{'_quick' if QUICK_RUN else ''}.npz"
    if path.exists():
        d = np.load(path)
        if np.array_equal(d["indices"], indices):
            return d["feats"], d["labels"]
    loader = DataLoader(Common224Dataset(train_ds, indices), batch_size=128, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    feats, labels = extract_features_from_loader(backbones[name], loader, DEVICE)
    np.savez(path, feats=feats, labels=labels, indices=indices)
    return feats, labels

heads, head_rows, head_histories = {}, [], {}
for name in BACKBONES:
    Xtr, ytr = cached_features(name, "train", train_idx)
    Xva, yva = cached_features(name, "val", val_idx)
    head, hist, best = train_linear_head(Xtr, ytr, Xva, yva, N_CLASSES, DEVICE, seed=SEED)
    heads[name] = head
    head_histories[name] = hist
    torch.save(head.state_dict(), CKPT / f"head_{name}.pt")
    head_rows.append({"backbone": name, "feature_dim": Xtr.shape[1], "epochs_run": len(hist),
                      "best_epoch": int(np.argmax([h["val_acc"] for h in hist])), "best_val_accuracy": best})
save_json(head_histories, RESULTS / "head_training_history.json")
display(save_table(head_rows, RESULTS / "head_training"))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for name, hist in head_histories.items():
    axes[0].plot([h["train_loss"] for h in hist], label=f"{name} train")
    axes[0].plot([h["val_loss"] for h in hist], "--", label=f"{name} val")
    axes[1].plot([h["val_acc"] for h in hist], label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy"); axes[0].legend(fontsize=6)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val accuracy"); axes[1].legend(fontsize=7)
fig.tight_layout()
from common.plotting import save_figure
save_figure(fig, figure_path("task1_head_training.png"))

/workspace/pa1-beyond-iid/task1/models/backbones.py:180: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * len(idx)


,backbone,feature_dim,epochs_run,best_epoch,best_val_accuracy
0,resnet50,2048,12,6,0.983
1,vit_b_16,768,9,3,0.988
2,clip_vit_b_32,512,7,1,0.996


PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_head_training.png')

In [4]:
from task1.analysis.feature_similarity import ModelSuite, MODEL_KEYS, MODEL_TO_BACKBONE
suite = ModelSuite(backbones, heads, CLASSES, DEVICE)
print("CLIP logit scale:", suite.logit_scale)

def evaluate(pils):
    return suite.run(pil_list_to_tensor(pils))

PRED_STORE = {}   # every prediction array, saved to task1/results/predictions.npz at the end
def store(tag, res):
    for m, out in res["outputs"].items():
        PRED_STORE[f"{tag}__{m}__preds"] = out["preds"]
        PRED_STORE[f"{tag}__{m}__maxprob"] = out["probs"].max(1)

CLIP logit scale: 100.0


## Step 1 — Clean baseline
Top-1 accuracy, macro-F1 and mean maximum confidence on the 500 clean images. Zero-shot CLIP confidence is the softmax over `logit_scale × cosine(image, "a photo of a {class}.")`.

In [5]:
from task1.analysis.evaluate_bias import (clean_baseline_rows, intervention_rows, translation_rows,
                                          shape_texture_rows, shape_texture_by_group, decision_category)
clean = evaluate(clean_pils)
store("clean", clean)
display(save_table(clean_baseline_rows(clean["outputs"], y), RESULTS / "step1_clean_baseline"))

per_class = []
for m, out in clean["outputs"].items():
    for k, c in enumerate(CLASSES):
        per_class.append({"model": m, "class": c, "accuracy": float((out["preds"][y == k] == k).mean())})
display(save_table(per_class, RESULTS / "step1_clean_per_class_accuracy").pivot(index="class", columns="model", values="accuracy"))

,model,top1_accuracy,macro_f1,mean_max_confidence
0,resnet50,0.980,0.9800,0.9757
1,vit_b_16,0.978,0.9780,0.9714
2,clip_head,0.988,0.9880,0.9637
3,clip_zeroshot,0.974,0.9739,0.9509


model,clip_head,clip_zeroshot,resnet50,vit_b_16
class,,,,
airplane,1.00,1.00,1.00,0.98
bird,1.00,1.00,0.96,0.94
car,0.94,0.94,0.96,0.98
cat,0.98,0.88,0.96,0.96
deer,1.00,0.96,0.96,0.96
dog,0.98,1.00,0.98,1.00
horse,0.98,0.98,1.00,0.98
monkey,1.00,0.98,1.00,1.00
ship,1.00,1.00,1.00,1.00


## Step 2 — Colour bias: grayscale (common) + luma-preserving hue rotation by 180° (chosen)
Grayscale tests the effect of *removing* chroma; hue rotation keeps luma, edges, texture contrast and saturation identical but makes the chroma *misleading*. A model that relies on colour should lose more accuracy under hue rotation than under grayscale.

In [6]:
from task1.data.transforms import to_grayscale_3ch, hue_rotate

gray_pils = [to_grayscale_3ch(im) for im in clean_pils]
hue_pils = [hue_rotate(im, 180.0) for im in clean_pils]
gray, hue = evaluate(gray_pils), evaluate(hue_pils)
store("grayscale", gray); store("hue_rot_180", hue)
color_rows = intervention_rows(clean["outputs"], gray["outputs"], y, "grayscale") + \
             intervention_rows(clean["outputs"], hue["outputs"], y, "hue_rot_180")
display(save_table(color_rows, RESULTS / "step2_color_bias"))

# Supplementary: accuracy / consistency as a function of the hue angle.
hue_sweep = []
for deg in [0, 45, 90, 135, 180]:
    res = clean if deg == 0 else (hue if deg == 180 else evaluate([hue_rotate(im, deg) for im in clean_pils]))
    for r in intervention_rows(clean["outputs"], res["outputs"], y, f"hue_{deg}"):
        hue_sweep.append({**r, "degrees": deg})
hue_df = save_table(hue_sweep, RESULTS / "step2_hue_angle_sweep")
from task1.analysis.figures import curves, MODEL_LABELS
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.1))
for metric, ax in zip(["accuracy", "prediction_consistency"], axes):
    series = {m: g.sort_values("degrees")[metric].values for m, g in hue_df.groupby("model", sort=False)}
    curves([0, 45, 90, 135, 180], series, metric.replace("_", " "), "Hue rotation (degrees)", None, ax=ax)
axes[0].legend(fontsize=7); fig.tight_layout()
save_figure(fig, figure_path("task1_hue_angle_sweep.png"))

,model,intervention,clean_accuracy,accuracy,accuracy_change,relative_accuracy,macro_f1,prediction_consistency,mean_max_confidence,clean_mean_max_confidence
0,resnet50,grayscale,0.980,0.940,-0.040,0.9592,0.9402,0.952,0.9632,0.9757
1,vit_b_16,grayscale,0.978,0.966,-0.012,0.9877,0.9661,0.970,0.9610,0.9714
2,clip_head,grayscale,0.988,0.960,-0.028,0.9717,0.9600,0.964,0.9403,0.9637
3,clip_zeroshot,grayscale,0.974,0.946,-0.028,0.9713,0.9457,0.964,0.9395,0.9509
4,resnet50,hue_rot_180,0.980,0.956,-0.024,0.9755,0.9561,0.964,0.9738,0.9757
5,vit_b_16,hue_rot_180,0.978,0.956,-0.022,0.9775,0.9562,0.958,0.9486,0.9714
6,clip_head,hue_rot_180,0.988,0.962,-0.026,0.9737,0.9621,0.964,0.9287,0.9637
7,clip_zeroshot,hue_rot_180,0.974,0.956,-0.018,0.9815,0.9559,0.968,0.9335,0.9509


PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_hue_angle_sweep.png')

## Step 3 — Shape vs. texture cue conflicts (AdaIN)
1. The (content, style) pairing of every candidate is fixed first (seed 6304): content images come from the evaluation subset, style images from test images **outside** the subset (central 60 % crop to emphasise object texture).
2. Candidates are stylized with the public pretrained AdaIN encoder/decoder (α = 1).
3. The pixel-only rejection rule is applied and the first 20 accepted images per direction are kept. The contact sheet below is produced **before** any model sees a cue-conflict image.

In [7]:
from task1.data.make_cue_conflicts import (AdaIN, CUE_CONFLICT_PAIRS, REJECTION_RULE, plan_cue_conflicts,
                                           stylization_metrics, accept_stylization, select_balanced, style_crop)

PER_DIRECTION = 5 if QUICK_RUN else 20
N_CANDIDATES = 8 if QUICK_RUN else 50   # all 50 content images of each class
style_pool_idx = np.setdiff1d(np.arange(len(test_labels)), subset_idx)
style_pool_labels = test_labels[style_pool_idx]
specs = plan_cue_conflicts(CLASSES, y, style_pool_labels, CUE_CONFLICT_PAIRS, N_CANDIDATES, seed=SEED)

adain = AdaIN(weights_dir=ROOT / "task1" / "checkpoints" / "adain").to(DEVICE)  # shared weight cache
style_pils = {}
def style_image(pos):
    if pos not in style_pils:
        style_pils[pos] = style_crop(to_common_224(stl10_pil(test_ds, int(style_pool_idx[pos]))))
    return style_pils[pos]

def render(specs, alpha, batch=16):
    out = []
    for i in range(0, len(specs), batch):
        chunk = specs[i:i + batch]
        c = pil_list_to_tensor([clean_pils[s["content_pos"]] for s in chunk]).to(DEVICE)
        s_ = pil_list_to_tensor([style_image(s["style_pool_pos"]) for s in chunk]).to(DEVICE)
        o = adain.stylize(c, s_, alpha=alpha)
        out.extend((o.permute(0, 2, 3, 1).cpu().numpy() * 255 + 0.5).astype(np.uint8))
    return out

ALPHA = 1.0
stylized = render(specs, ALPHA)
decisions = []
for s, img in zip(specs, stylized):
    met = stylization_metrics(np.asarray(clean_pils[s["content_pos"]]), img)
    ok, reason = accept_stylization(met)
    decisions.append((ok, reason, met))
kept = select_balanced(specs, [d[0] for d in decisions], per_direction=PER_DIRECTION)
kept_set = set(kept)

conflict_dir = CACHE / "cue_conflicts"; conflict_dir.mkdir(exist_ok=True)
manifest = []
for i, (s, (ok, reason, met)) in enumerate(zip(specs, decisions)):
    fname = f"{i:04d}_{s['content_class']}_as_{s['style_class']}.png"
    if i in kept_set:
        Image.fromarray(stylized[i]).save(conflict_dir / fname)
    manifest.append({**s, "accepted": ok, "reason": reason, **met, "kept": i in kept_set,
                     "file": fname if i in kept_set else None,
                     "content_test_index": int(subset_idx[s["content_pos"]]),
                     "style_test_index": int(style_pool_idx[s["style_pool_pos"]])})
save_json({"alpha": ALPHA, "rejection_rule": REJECTION_RULE, "per_direction": PER_DIRECTION,
           "candidates": manifest}, RESULTS / "step3_cue_conflict_manifest.json")

man = pd.DataFrame(manifest)
counts = man.groupby(["pair", "direction"]).agg(generated=("accepted", "size"), accepted=("accepted", "sum"),
                                                 kept=("kept", "sum")).reset_index()
reasons = man.groupby(["direction", "reason"]).size().unstack(fill_value=0).add_prefix("n_").reset_index()
counts = counts.merge(reasons, on="direction")
display(save_table(counts.to_dict("records"), RESULTS / "step3_accept_reject_counts"))
print(f"generated {len(man)} | accepted {int(man.accepted.sum())} | rejected {int((~man.accepted).sum())} | kept {len(kept)}")

,pair,direction,generated,accepted,kept,n_accepted,n_content_shape_destroyed
0,airplane-cat,shape=airplane/texture=cat,50,40,20,40,10
1,airplane-cat,shape=cat/texture=airplane,50,44,20,44,6
2,bird-car,shape=bird/texture=car,50,44,20,44,6
3,bird-car,shape=car/texture=bird,50,43,20,43,7
4,deer-truck,shape=deer/texture=truck,50,46,20,46,4
5,deer-truck,shape=truck/texture=deer,50,44,20,44,6
6,dog-ship,shape=dog/texture=ship,50,48,20,48,2
7,dog-ship,shape=ship/texture=dog,50,35,20,35,15
8,horse-airplane,shape=airplane/texture=horse,50,33,20,33,17
9,horse-airplane,shape=horse/texture=airplane,50,44,20,44,6


generated 600 | accepted 503 | rejected 97 | kept 240


In [8]:
# Contact sheet (made before evaluation): accepted examples and one example per rejection reason.
from task1.analysis.figures import cue_conflict_gallery
gal = []
for direction in list(dict.fromkeys(man["direction"]))[:8]:
    idx = man.index[(man.direction == direction) & man.kept]
    if len(idx) == 0:
        continue
    i = idx[0]
    s = specs[i]
    gal.append({"content": np.asarray(clean_pils[s["content_pos"]]), "style": np.asarray(style_image(s["style_pool_pos"])),
                "stylized": stylized[i], "title": f"ACCEPT {s['content_class']}/{s['style_class']}",
                "caption": f"edge r={man.edge_correlation[i]:.2f}"})
for reason in ["content_shape_destroyed", "style_not_transferred", "flat_output"]:
    idx = man.index[man.reason == reason]
    for i in idx[:2]:
        s = specs[i]
        gal.append({"content": np.asarray(clean_pils[s["content_pos"]]), "style": np.asarray(style_image(s["style_pool_pos"])),
                    "stylized": stylized[i], "title": f"REJECT ({reason})",
                    "caption": f"edge r={man.edge_correlation[i]:.2f}, diff={man.mean_abs_diff_to_content[i]:.2f}"})
cue_conflict_gallery(gal, figure_path("task1_cue_conflict_contact_sheet.png"), n_cols=4)

PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_cue_conflict_contact_sheet.png')

In [9]:
# Model evaluation on the kept cue conflicts.
kspecs = [specs[i] for i in kept]
conflict_pils = [Image.fromarray(stylized[i]) for i in kept]
shape_y = np.array([CLASSES.index(s["content_class"]) for s in kspecs])
texture_y = np.array([CLASSES.index(s["style_class"]) for s in kspecs])
conflict = evaluate(conflict_pils)
store("cue_conflict", conflict)
PRED_STORE["cue_conflict__shape_labels"], PRED_STORE["cue_conflict__texture_labels"] = shape_y, texture_y

sb_rows = shape_texture_rows(conflict["outputs"], shape_y, texture_y)
display(save_table(sb_rows, RESULTS / "step3_shape_bias"))
display(save_table(shape_texture_by_group(conflict["outputs"], shape_y, texture_y, [s["pair"] for s in kspecs]),
                   RESULTS / "step3_shape_bias_by_pair").pivot(index="group", columns="model", values="shape_bias_pct"))
save_table(shape_texture_by_group(conflict["outputs"], shape_y, texture_y, [s["direction"] for s in kspecs]),
           RESULTS / "step3_shape_bias_by_direction")

# Content-image accuracy on exactly the same images (how often the clean content was recognised).
content_pos = np.array([s["content_pos"] for s in kspecs])
display(pd.DataFrame([{"model": m, "clean_content_accuracy": float((clean["outputs"][m]["preds"][content_pos] == shape_y).mean())}
                      for m in MODEL_KEYS]))
from task1.analysis.figures import decision_bars
decision_bars(sb_rows, figure_path("task1_shape_texture_decisions.png"))

,model,n_shape,n_texture,n_other,n_total,shape_bias_pct,coverage_pct,mean_max_confidence
0,resnet50,47,44,149,240,51.6484,37.9167,0.8554
1,vit_b_16,102,23,115,240,81.6000,52.0833,0.6366
2,clip_head,106,9,125,240,92.1739,47.9167,0.5071
3,clip_zeroshot,115,14,111,240,89.1473,53.7500,0.5778


model,clip_head,clip_zeroshot,resnet50,vit_b_16
group,,,,
airplane-cat,100.0000,94.1176,66.6667,70.8333
bird-car,90.0000,96.5517,61.5385,86.2069
deer-truck,92.3077,100.0000,50.0000,88.8889
dog-ship,87.5000,60.0000,21.0526,68.4211
horse-airplane,88.8889,85.0000,71.4286,80.0000
monkey-car,95.0000,95.2381,63.6364,95.0000


,model,clean_content_accuracy
0,resnet50,0.9833
1,vit_b_16,0.9792
2,clip_head,0.9875
3,clip_zeroshot,0.9708


PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_shape_texture_decisions.png')

In [10]:
# Informative agreements / disagreements / failures (selection for display only).
cats = {m: decision_category(conflict["outputs"][m]["preds"], shape_y, texture_y) for m in MODEL_KEYS}
cat_mat = np.stack([cats[m] for m in MODEL_KEYS], 1)
groups = {
    "all shape": np.where((cat_mat == "shape").all(1))[0],
    "all texture": np.where((cat_mat == "texture").all(1))[0],
    "CNN texture / CLIP shape": np.where((cats["resnet50"] == "texture") & (cats["clip_zeroshot"] == "shape"))[0],
    "ViT texture / CNN shape": np.where((cats["vit_b_16"] == "texture") & (cats["resnet50"] == "shape"))[0],
    "head vs zero-shot disagree": np.where(cats["clip_head"] != cats["clip_zeroshot"])[0],
    "all other": np.where((cat_mat == "other").all(1))[0],
}
abbrev = {"resnet50": "R50", "vit_b_16": "ViT", "clip_head": "CLIPh", "clip_zeroshot": "CLIPzs"}
rng = np.random.RandomState(SEED)
examples, example_rows = [], []
for g, idx in groups.items():
    for i in rng.permutation(idx)[:2]:
        s = kspecs[i]
        preds = {m: CLASSES[conflict["outputs"][m]["preds"][i]] for m in MODEL_KEYS}
        cap = " ".join(f"{abbrev[m]}:{preds[m]}" for m in MODEL_KEYS)
        examples.append({"content": np.asarray(clean_pils[s["content_pos"]]), "style": np.asarray(style_image(s["style_pool_pos"])),
                         "stylized": np.asarray(conflict_pils[i]), "title": f"{g}: shape={s['content_class']} tex={s['style_class']}",
                         "caption": cap})
        example_rows.append({"group": g, "file": manifest[kept[i]]["file"], "shape": s["content_class"],
                             "texture": s["style_class"], **preds})
print({g: len(v) for g, v in groups.items()})
save_table(example_rows, RESULTS / "step3_cue_conflict_examples")
cue_conflict_gallery(examples, figure_path("task1_cue_conflict_examples.png"), n_cols=3)

{'all shape': 31, 'all texture': 0, 'CNN texture / CLIP shape': 17, 'ViT texture / CNN shape': 2, 'head vs zero-shot disagree': 49, 'all other': 50}


PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_cue_conflict_examples.png')

In [11]:
# Supplementary: style-strength sensitivity on the same kept (content, style) pairs.
alpha_rows = [dict(r, alpha=ALPHA) for r in sb_rows]
for a in [0.5, 0.75]:
    imgs = render(kspecs, a)
    out = evaluate([Image.fromarray(im) for im in imgs])
    alpha_rows += [dict(r, alpha=a) for r in shape_texture_rows(out["outputs"], shape_y, texture_y)]
alpha_df = save_table(alpha_rows, RESULTS / "step3_style_strength_sweep")
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.1))
for metric, ax in zip(["shape_bias_pct", "coverage_pct"], axes):
    series = {m: g.sort_values("alpha")[metric].values for m, g in alpha_df.groupby("model", sort=False)}
    curves([0.5, 0.75, 1.0], series, metric.replace("_pct", " (%)").replace("_", " "), "AdaIN style strength α", None, ax=ax)
axes[0].legend(fontsize=7); fig.tight_layout()
save_figure(fig, figure_path("task1_style_strength_sweep.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_style_strength_sweep.png')

## Step 4 — Translation (0/8/16/32 px, four cardinal directions, reflection padding + shifted crop)
Accuracy and consistency are computed separately for each direction and then averaged across directions (error bars = std over directions).

In [12]:
from task1.data.transforms import translate_reflect, TRANSLATION_PIXELS, TRANSLATION_DIRECTIONS
shifted = {}
for p in TRANSLATION_PIXELS[1:]:
    for d in TRANSLATION_DIRECTIONS:
        shifted[(p, d)] = evaluate([translate_reflect(im, p, d) for im in clean_pils])
        store(f"translate_{p}_{d}", shifted[(p, d)])
trans_df = save_table(translation_rows(clean["outputs"], {k: v["outputs"] for k, v in shifted.items()}, y),
                      RESULTS / "step4_translation")
trans_mean = trans_df[trans_df.direction == "mean"]
display(trans_mean.pivot(index="pixels", columns="model", values="accuracy"))
display(trans_mean.pivot(index="pixels", columns="model", values="consistency"))
from task1.analysis.figures import translation_figure
translation_figure(trans_mean, figure_path("task1_translation.png"))

model,clip_head,clip_zeroshot,resnet50,vit_b_16
pixels,,,,
0,0.9880,0.9740,0.9800,0.9780
8,0.9820,0.9735,0.9815,0.9815
16,0.9815,0.9730,0.9810,0.9770
32,0.9740,0.9620,0.9740,0.9735


model,clip_head,clip_zeroshot,resnet50,vit_b_16
pixels,,,,
0,1.0000,1.0000,1.0000,1.0000
8,0.9885,0.9775,0.9890,0.9870
16,0.9850,0.9770,0.9880,0.9955
32,0.9840,0.9795,0.9805,0.9885


PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_translation.png')

## Step 5 — Patch structure (4×4 pixel grid, one non-identity permutation per image, seed 6304)
The same shuffled images are given to every model. Because a confident prediction on a shuffled image is not necessarily sensible, the confidence and the distribution of predicted classes are recorded too.

In [13]:
from task1.data.transforms import non_identity_permutations, patch_shuffle
perms = non_identity_permutations(len(clean_pils), grid=4, seed=SEED)
save_json({"grid": 4, "seed": SEED, "permutations": perms.tolist()}, RESULTS / "step5_patch_permutations.json")
patch_pils = [patch_shuffle(im, perm, grid=4) for im, perm in zip(clean_pils, perms)]
patch = evaluate(patch_pils)
store("patch_shuffle_4x4", patch)
patch_rows = intervention_rows(clean["outputs"], patch["outputs"], y, "patch_shuffle_4x4")
display(save_table(patch_rows, RESULTS / "step5_patch_shuffle"))

# Supplementary: grid-size sweep (2x2 = coarse, 8x8 = fine shuffling).
grid_rows = [dict(r, grid=4) for r in patch_rows]
for g in [2, 8]:
    pg = non_identity_permutations(len(clean_pils), grid=g, seed=SEED)
    res = evaluate([patch_shuffle(im, perm, grid=g) for im, perm in zip(clean_pils, pg)])
    grid_rows += [dict(r, grid=g) for r in intervention_rows(clean["outputs"], res["outputs"], y, f"patch_shuffle_{g}x{g}")]
grid_df = save_table(grid_rows, RESULTS / "step5_patch_grid_sweep")
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.1))
for metric, ax in zip(["accuracy", "prediction_consistency"], axes):
    series = {m: g.sort_values("grid")[metric].values for m, g in grid_df.groupby("model", sort=False)}
    curves([2, 4, 8], series, metric.replace("_", " "), "Patch grid (g×g)", None, ax=ax)
axes[0].legend(fontsize=7); fig.tight_layout()
save_figure(fig, figure_path("task1_patch_grid_sweep.png"))

# Which classes absorb shuffled images, and how confident are the models?
dist_rows = []
for m in MODEL_KEYS:
    for k, c in enumerate(CLASSES):
        dist_rows.append({"model": m, "class": c,
                          "clean_pred_frac": float((clean["outputs"][m]["preds"] == k).mean()),
                          "shuffled_pred_frac": float((patch["outputs"][m]["preds"] == k).mean())})
dist_df = save_table(dist_rows, RESULTS / "step5_predicted_class_distribution")
display(dist_df.pivot(index="class", columns="model", values="shuffled_pred_frac"))
fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), sharey=True)
for ax, m in zip(axes, MODEL_KEYS):
    ax.hist(patch["outputs"][m]["probs"].max(1), bins=20, range=(0, 1), alpha=0.6, label="shuffled")
    ax.hist(clean["outputs"][m]["probs"].max(1), bins=20, range=(0, 1), alpha=0.6, label="clean")
    ax.set_title(MODEL_LABELS[m], fontsize=9); ax.set_xlabel("max softmax probability")
axes[0].legend(fontsize=7); fig.tight_layout()
save_figure(fig, figure_path("task1_patch_shuffle_confidence.png"))

# Most confident wrong predictions on shuffled images, per model.
from task1.analysis.figures import image_grid
ims, tts = [], []
for m in MODEL_KEYS:
    o = patch["outputs"][m]
    wrong = np.where(o["preds"] != y)[0]
    for i in wrong[np.argsort(-o["probs"].max(1)[wrong])][:3]:
        ims.append(np.asarray(patch_pils[i]))
        tts.append(f"{abbrev[m]}: {CLASSES[o['preds'][i]]} ({o['probs'].max(1)[i]:.2f})\ntrue {CLASSES[y[i]]}")
image_grid(ims, tts, figure_path("task1_patch_shuffle_failures.png"), n_cols=6)

,model,intervention,clean_accuracy,accuracy,accuracy_change,relative_accuracy,macro_f1,prediction_consistency,mean_max_confidence,clean_mean_max_confidence
0,resnet50,patch_shuffle_4x4,0.980,0.892,-0.088,0.9102,0.8918,0.896,0.9353,0.9757
1,vit_b_16,patch_shuffle_4x4,0.978,0.916,-0.062,0.9366,0.9159,0.926,0.8957,0.9714
2,clip_head,patch_shuffle_4x4,0.988,0.850,-0.138,0.8603,0.8502,0.852,0.7700,0.9637
3,clip_zeroshot,patch_shuffle_4x4,0.974,0.840,-0.134,0.8624,0.8387,0.840,0.8339,0.9509


model,clip_head,clip_zeroshot,resnet50,vit_b_16
class,,,,
airplane,0.094,0.088,0.084,0.102
bird,0.142,0.134,0.100,0.100
car,0.110,0.124,0.116,0.100
cat,0.094,0.082,0.100,0.096
deer,0.092,0.108,0.122,0.108
dog,0.070,0.118,0.092,0.108
horse,0.088,0.092,0.100,0.090
monkey,0.124,0.072,0.084,0.100
ship,0.084,0.100,0.108,0.100


PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_patch_shuffle_failures.png')

## Compact comparison (required evidence 1)

In [14]:
compact = []
for m in MODEL_KEYS:
    row = {"model": m, "clean_acc": float((clean["outputs"][m]["preds"] == y).mean())}
    for r in color_rows + patch_rows:
        if r["model"] == m:
            row[f"{r['intervention']}_acc"] = r["accuracy"]
            row[f"{r['intervention']}_delta"] = r["accuracy_change"]
            row[f"{r['intervention']}_consistency"] = r["prediction_consistency"]
    compact.append(row)
display(save_table(compact, RESULTS / "summary_clean_color_patch"))

,model,clean_acc,grayscale_acc,grayscale_delta,grayscale_consistency,hue_rot_180_acc,hue_rot_180_delta,hue_rot_180_consistency,patch_shuffle_4x4_acc,patch_shuffle_4x4_delta,patch_shuffle_4x4_consistency
0,resnet50,0.980,0.940,-0.040,0.952,0.956,-0.024,0.964,0.892,-0.088,0.896
1,vit_b_16,0.978,0.966,-0.012,0.970,0.956,-0.022,0.958,0.916,-0.062,0.926
2,clip_head,0.988,0.960,-0.028,0.964,0.962,-0.026,0.964,0.850,-0.138,0.852
3,clip_zeroshot,0.974,0.946,-0.028,0.964,0.956,-0.018,0.968,0.840,-0.134,0.840


## Step 6 — Representation analysis
Cosine stability $I_T$ of each backbone's representation, for grayscale, hue rotation, cue conflict (paired with its content image), translation (each displacement, averaged over the four directions) and patch shuffling. Because raw cosine levels differ across backbones (ResNet features are non-negative, CLIP embeddings are normalised), each row also reports the mean cosine between *unrelated* clean images and a normalised stability $(I_T - \text{floor})/(1-\text{floor})$. $I_T$ is also split by whether the prediction changed, to compare prediction-level and representation-level effects.

In [15]:
from task1.analysis.representation import stability_rows, fit_2d_projection
BACKBONE_MODELS = {"resnet50": ["resnet50"], "vit_b_16": ["vit_b_16"], "clip_vit_b_32": ["clip_head", "clip_zeroshot"]}

def flips(clean_preds_by_model, trans_outputs):
    return {m: clean_preds_by_model[m] != trans_outputs[m]["preds"] for m in clean_preds_by_model}

interventions = {  # name -> (clean feats index, transformed result)
    "grayscale": (np.arange(len(y)), gray),
    "hue_rot_180": (np.arange(len(y)), hue),
    "cue_conflict": (content_pos, conflict),
    "patch_shuffle_4x4": (np.arange(len(y)), patch),
}
stab_rows, pred_rep_rows = [], []
for bb, models in BACKBONE_MODELS.items():
    for name, (cidx, res) in interventions.items():
        rows = stability_rows(bb, name, clean["feats"][bb][cidx], res["feats"][bb],
                              flips({m: clean["outputs"][m]["preds"][cidx] for m in models}, res["outputs"]), SEED)
        stab_rows.append(rows[0]); pred_rep_rows += rows[1:]
    for p in TRANSLATION_PIXELS[1:]:
        ct = np.concatenate([clean["feats"][bb]] * 4)
        tt = np.concatenate([shifted[(p, d)]["feats"][bb] for d in TRANSLATION_DIRECTIONS])
        fl = {m: np.concatenate([clean["outputs"][m]["preds"] != shifted[(p, d)]["outputs"][m]["preds"]
                                 for d in TRANSLATION_DIRECTIONS]) for m in models}
        rows = stability_rows(bb, f"translation_{p}", ct, tt, fl, SEED)
        stab_rows.append(rows[0]); pred_rep_rows += rows[1:]
stab_df = save_table(stab_rows, RESULTS / "step6_representation_stability")
display(stab_df.pivot(index="intervention", columns="backbone", values="I_T"))
display(stab_df.pivot(index="intervention", columns="backbone", values="normalized_stability"))
display(save_table(pred_rep_rows, RESULTS / "step6_prediction_vs_representation"))

backbone,clip_vit_b_32,resnet50,vit_b_16
intervention,,,
cue_conflict,0.6205,0.2194,0.1825
grayscale,0.9029,0.7914,0.7481
hue_rot_180,0.9056,0.8343,0.7108
patch_shuffle_4x4,0.7314,0.7171,0.6575
translation_16,0.9675,0.9532,0.9729
translation_32,0.9696,0.9333,0.9414
translation_8,0.9808,0.9690,0.9473


backbone,clip_vit_b_32,resnet50,vit_b_16
intervention,,,
cue_conflict,0.0301,-0.0465,0.0958
grayscale,0.7475,0.7209,0.7220
hue_rot_180,0.7543,0.7782,0.6808
patch_shuffle_4x4,0.3012,0.6214,0.6220
translation_16,0.9154,0.9374,0.9702
translation_32,0.9210,0.9109,0.9356
translation_8,0.9499,0.9585,0.9421


,backbone,intervention,model,frac_prediction_changed,I_T_prediction_unchanged,I_T_prediction_changed
0,resnet50,grayscale,resnet50,0.0480,0.7958,0.7045
1,resnet50,hue_rot_180,resnet50,0.0360,0.8375,0.7490
2,resnet50,cue_conflict,resnet50,0.8042,0.3621,0.1847
3,resnet50,patch_shuffle_4x4,resnet50,0.1040,0.7190,0.7004
4,resnet50,translation_8,resnet50,0.0110,0.9691,0.9578
5,resnet50,translation_16,resnet50,0.0120,0.9535,0.9257
6,resnet50,translation_32,resnet50,0.0195,0.9341,0.8975
7,vit_b_16,grayscale,vit_b_16,0.0300,0.7522,0.6155
8,vit_b_16,hue_rot_180,vit_b_16,0.0420,0.7191,0.5217
9,vit_b_16,cue_conflict,vit_b_16,0.5667,0.2858,0.1035


In [16]:
# Prediction consistency vs. normalized representation stability (one point per model x intervention).
from task1.analysis.figures import stability_vs_consistency, grouped_bars
cons_lookup = {}
for r in color_rows + patch_rows:
    cons_lookup[(r["model"], r["intervention"])] = r["prediction_consistency"]
for _, r in trans_mean.iterrows():
    if r["pixels"] > 0:
        cons_lookup[(r["model"], f"translation_{int(r['pixels'])}")] = r["consistency"]
for m in MODEL_KEYS:  # for cue conflicts, "consistency" = prediction unchanged w.r.t. the clean content image
    cons_lookup[(m, "cue_conflict")] = float((conflict["outputs"][m]["preds"] == clean["outputs"][m]["preds"][content_pos]).mean())
points = []
for r in stab_rows:
    for m in BACKBONE_MODELS[r["backbone"]]:
        if (m, r["intervention"]) in cons_lookup:
            points.append({"model": m, "intervention": r["intervention"], "normalized_stability": r["normalized_stability"],
                           "consistency": cons_lookup[(m, r["intervention"])]})
save_table(points, RESULTS / "step6_stability_vs_consistency")
stability_vs_consistency(points, figure_path("task1_stability_vs_consistency.png"))
grouped_bars(stab_df, "intervention", "backbone", "normalized_stability", figure_path("task1_representation_stability.png"),
             "normalized cosine stability", labels={"resnet50": "ResNet-50", "vit_b_16": "ViT-B/16", "clip_vit_b_32": "CLIP ViT-B/32"})

PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_representation_stability.png')

In [17]:
# CLIP zero-shot vs. its trained linear head, on the same embedding.
zs_rows = []
for tag, res in [("clean", clean), ("grayscale", gray), ("hue_rot_180", hue), ("patch_shuffle_4x4", patch),
                 ("translation_32_right", shifted[(32, "right")])]:
    h, z = res["outputs"]["clip_head"]["preds"], res["outputs"]["clip_zeroshot"]["preds"]
    zs_rows.append({"condition": tag, "head_accuracy": float((h == y).mean()), "zeroshot_accuracy": float((z == y).mean()),
                    "head_zeroshot_agreement": float((h == z).mean())})
h, z = conflict["outputs"]["clip_head"]["preds"], conflict["outputs"]["clip_zeroshot"]["preds"]
zs_rows.append({"condition": "cue_conflict", "head_accuracy": float((h == shape_y).mean()),
                "zeroshot_accuracy": float((z == shape_y).mean()), "head_zeroshot_agreement": float((h == z).mean())})
display(save_table(zs_rows, RESULTS / "step6_clip_head_vs_zeroshot"))

,condition,head_accuracy,zeroshot_accuracy,head_zeroshot_agreement
0,clean,0.9880,0.9740,0.9780
1,grayscale,0.9600,0.9460,0.9600
2,hue_rot_180,0.9620,0.9560,0.9740
3,patch_shuffle_4x4,0.8500,0.8400,0.8680
4,translation_32_right,0.9720,0.9580,0.9560
5,cue_conflict,0.4417,0.4792,0.6083


### t-SNE of clean vs. transformed representations
One projection per (backbone, intervention), fitted on the concatenation of clean and transformed features of the same images (settings saved to `step6_tsne_settings.json`). Colour = ground-truth class (for cue conflicts: the content/shape class), marker = clean (●) vs. transformed (×). Coordinates are not comparable across panels.

In [18]:
from task1.analysis.figures import tsne_grid
TSNE_SETTINGS = {"method": "tsne", "metric": "cosine", "perplexity": 30, "init": "pca", "seed": SEED,
                 "translation_panel": "32 px, direction assigned cyclically per image (up, down, left, right)"}
save_json(TSNE_SETTINGS, RESULTS / "step6_tsne_settings.json")
cyc_dir = [TRANSLATION_DIRECTIONS[i % 4] for i in range(len(y))]
panels = {}
for bb in BACKBONE_MODELS:
    t32 = np.stack([shifted[(32, d)]["feats"][bb][i] for i, d in enumerate(cyc_dir)])
    for col, (cf, tf, lab) in {
        "Grayscale": (clean["feats"][bb], gray["feats"][bb], y),
        "Cue conflict (α=1)": (clean["feats"][bb][content_pos], conflict["feats"][bb], shape_y),
        "Translation 32 px": (clean["feats"][bb], t32, y),
        "Patch shuffle 4×4": (clean["feats"][bb], patch["feats"][bb], y),
    }.items():
        xy = fit_2d_projection(np.concatenate([cf, tf]), method="tsne", seed=SEED, perplexity=30)
        panels[(bb, col)] = (xy, np.concatenate([lab, lab]), np.r_[np.zeros(len(cf), bool), np.ones(len(tf), bool)])
tsne_grid(panels, CLASSES, figure_path("task1_tsne_grid.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task1_tsne_grid.png')

In [19]:
# Illustration of every intervention on a few images (for the report's method figure).
from task1.analysis.figures import intervention_examples
show = [int(np.where(y == k)[0][0]) for k in [0, 3, 8]]
rows = []
for i in show:
    rows.append([np.asarray(clean_pils[i]), np.asarray(gray_pils[i]), np.asarray(hue_pils[i]),
                 np.asarray(translate_reflect(clean_pils[i], 32, "right")), np.asarray(patch_pils[i])])
intervention_examples(rows, [CLASSES[y[i]] for i in show],
                      ["clean", "grayscale", "hue +180°", "shift 32 px right", "patch shuffle 4×4"],
                      figure_path("task1_interventions.png"))

np.savez_compressed(RESULTS / "predictions.npz", labels=y, subset_idx=subset_idx, **PRED_STORE)
print("saved", len(PRED_STORE), "prediction arrays")

saved 138 prediction arrays


## Research questions (answer in your own words in the report)
1. What do the colour and cue-conflict experiments jointly reveal about each model's reliance on shape, texture and colour? How does coverage affect the strength of the shape-bias conclusions? → `step2_color_bias`, `step2_hue_angle_sweep`, `step3_shape_bias*`, `step3_style_strength_sweep`.
2. How do ResNet, ViT and CLIP respond to translation and patch shuffling? → `step4_translation`, `step5_patch_shuffle`, `step5_patch_grid_sweep`, `step5_predicted_class_distribution`.
3. To what extent do prediction-level changes agree with representation changes? CLIP zero-shot vs. head? → `step6_*`.
4. Which differences are attributable to architecture vs. pretraining data / supervision / augmentation / capacity?